# Assignment 1 — Do personas annotate like people, and does context change them?

**Student:** Ioannis Petridis (f3352601)
**Course:** Practical Data Science, MSc in Data Science, AUEB
**GitHub repository:** https://github.com/PetridhsG/practical-data-science-assignment-1
**Seed:** `542685`

## Project description

Toxicity is subjective: two careful readers can rate the same comment differently. In this notebook I rate
20 online comments for toxicity myself, and then ask a small LLM to rate the same comments while playing
annotators with different identities (personas), once with the surrounding conversation as context and once
without it. Comparing the ratings tells me whether any persona annotates like me, and whether context changes
how the personas judge toxicity.

## Setup

**Tools.** The conversations (part C) and the persona annotators (parts D and E) are run with
[SynDisco](https://pypi.org/project/syndisco/), a library for synthetic discussions and annotation with LLMs.
SynDisco loads the model through Hugging Face `transformers`, which it installs together with its other
dependencies. The model is a 4-bit quantised instruction-tuned model, so the notebook runs on a Google Colab
**T4 GPU**.

**Libraries.**

| Library | Used for |
|---|---|
| `syndisco` | actors (persona + context + instructions), discussions, annotation jobs |
| `torch` | runs the model on the GPU |
| `pandas` | the tables of items, ratings and agreements |
| `matplotlib` | histogram, boxplot and agreement heatmaps |
| `json`, `random`, `re`, `pathlib` (standard library) | saving conversations, random participants, parsing the model's answers, file paths |

Only `syndisco` has to be installed; `torch`, `pandas` and `matplotlib` are preinstalled on Colab.

In [ ]:
%pip install -q syndisco

In [ ]:
import json                      # to save the conversations
import random                    # to choose random participants
import re                        # to find the rating and evidence in the model's answer
from pathlib import Path         # file paths

import matplotlib.pyplot as plt  # for the plots
import pandas as pd              # for the tables
import torch                     # the library that runs the model
import syndisco                  # conversations and annotators

SEED = 542685                    # the seed of prepare_items.py, reused for every random choice
random.seed(SEED)
torch.manual_seed(SEED)

pd.set_option("display.max_colwidth", None)  # print the texts in full

print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none (Runtime > Change runtime type > T4 GPU)")

## A. My own annotation

I drew my sample with `python prepare_items.py --seed 542685`: 20 comments from news-site discussions,
a balanced mix of civil and toxic texts taken from a pool of 250. The script wrote `items.csv` and
`annotate.html`, the page I used for rating.

I rated each item on a 1-5 scale of how uncivil or hostile its language is, judging the tone and not the
topic (1-2 = civil, 3-5 = toxic). For ratings 3-5 I selected the words that make the text toxic as evidence.
I rated alone, before seeing any model output, and never changed a rating. The result is
`data/items_rated.csv` (id, text, rating, label, evidence).

## B. Exploratory data analysis

Both files are read directly from the `data/` folder of the GitHub repository, so the notebook needs no
local files.

In [ ]:
DATA_URL = "https://raw.githubusercontent.com/PetridhsG/practical-data-science-assignment-1/main/data/"

items = pd.read_csv(DATA_URL + "items.csv")              # the sample from prepare_items.py
rated = pd.read_csv(DATA_URL + "items_rated.csv")        # my ratings from annotate.html
print("items:", items.shape, " rated:", rated.shape)

The unrated sample, in full:

In [ ]:
items

My ratings, in full:

In [ ]:
rated